# LeetCode #146 — LRU Cache: hash map + danh sách liên kết đôi, O(1) mọi thao tác

**Mức độ:** Medium · **Chủ đề:** Hash Table, Linked List, Design, Doubly-Linked List

Skeleton trên LeetCode:

```python
class LRUCache:
    def __init__(self, capacity: int):

    def get(self, key: int) -> int:

    def put(self, key: int, value: int) -> None:
```

Mục tiêu của notebook:

1. Hiểu chính sách **LRU (Least Recently Used)**: khi đầy, bỏ khóa **lâu nhất chưa được dùng**.
2. Vì sao cần **hai** cấu trúc: hash map (tìm `O(1)`) + danh sách liên kết **đôi** (di chuyển / xóa `O(1)`).
3. Kỹ thuật **hai nút canh (sentinel)** `head`, `tail` để khỏi xử lý biên.
4. Cách gọn trong Python: `collections.OrderedDict` với `move_to_end` và `popitem(last=False)`.

# 1. Bài toán là gì?

Thiết kế cấu trúc dữ liệu bộ nhớ đệm LRU:

- `LRUCache(capacity)`: khởi tạo với dung lượng dương `capacity`.
- `get(key)`: trả giá trị nếu khóa tồn tại, ngược lại `-1`. Lần truy cập này làm khóa trở thành **mới dùng nhất**.
- `put(key, value)`: cập nhật nếu khóa đã có; ngược lại thêm mới. Nếu số khóa **vượt** `capacity`, bỏ khóa **lâu nhất chưa dùng**. `put` cũng tính là "dùng".

`get` và `put` phải chạy **`O(1)` trung bình**.

**Constraints (theo đề):**

- `1 <= capacity <= 3000`
- `0 <= key <= 10^4`, `0 <= value <= 10^5`
- Tối đa `2 * 10^5` lời gọi `get` và `put`.

---

## Ví dụ

```text
LRUCache(2)
put(1, 1)          cache = {1=1}
put(2, 2)          cache = {1=1, 2=2}
get(1)   -> 1      dùng 1 -> thứ tự (cũ → mới): 2, 1
put(3, 3)          đầy -> bỏ 2 (lâu nhất)  -> {1=1, 3=3}
get(2)   -> -1
put(4, 4)          bỏ 1 -> {3=3, 4=4}
get(1)   -> -1
get(3)   -> 3
get(4)   -> 4
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bảng băm (hash table) — `dict` và `set`
2. Bộ nhớ đệm (cache) và chính sách LRU

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [1]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [2]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 259.17 ms | trên set: 0.076 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [3]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [4]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.2 Bộ nhớ đệm (cache) và chính sách LRU

### Cache là gì? Vì sao cần "đuổi" phần tử?

**Cache** là vùng nhớ nhỏ, nhanh, giữ lại kết quả đã dùng để lần sau lấy ngay thay vì tính/tải lại (ví dụ: trình duyệt lưu ảnh của trang vừa xem). Vì cache có **sức chứa giới hạn** `capacity`, khi đầy mà cần thêm mục mới, ta phải **đuổi** (evict) một mục cũ. Chọn mục nào để đuổi gọi là **chính sách thay thế**:

| Chính sách | Đuổi mục nào |
|---|---|
| FIFO | mục được **thêm vào sớm nhất** |
| LFU | mục được dùng **ít lần nhất** |
| **LRU** (Least Recently Used) | mục **lâu nhất chưa được dùng** |

LRU dựa trên kinh nghiệm: thứ vừa dùng gần đây có khả năng sắp được dùng lại.

### Ví dụ LRU với `capacity = 2`

"Dùng" = `get` hoặc `put` một khóa. Ta giữ các khóa theo thứ tự **cũ nhất → mới nhất**:

```text
put(1)       [1]
put(2)       [1, 2]
get(1)       [2, 1]        ← 1 vừa dùng, chuyển về cuối
put(3)       đầy → đuổi 2 (cũ nhất) → [1, 3]
get(2)       -1 (đã bị đuổi)
put(4)       đầy → đuổi 1 → [3, 4]
```

### Mô phỏng ngây thơ bằng `list` (để hiểu, không để nộp)

Giữ danh sách khóa theo thứ tự dùng. Mỗi lần dùng: xóa khóa khỏi vị trí cũ (`O(n)`) rồi thêm vào cuối. Đuổi = bỏ phần tử đầu.

In [5]:
class LRUNgayTho:
    def __init__(self, capacity):
        self.cap, self.data, self.order = capacity, {}, []   # order: cũ nhất -> mới nhất

    def _dung(self, key):
        if key in self.order:
            self.order.remove(key)          # O(n)
        self.order.append(key)

    def get(self, key):
        if key not in self.data:
            return -1
        self._dung(key)
        return self.data[key]

    def put(self, key, value):
        if key not in self.data and len(self.data) == self.cap:
            cu_nhat = self.order.pop(0)     # O(n)
            del self.data[cu_nhat]
        self.data[key] = value
        self._dung(key)

c = LRUNgayTho(2)
nhat_ky = []
c.put(1, 1); c.put(2, 2); nhat_ky.append(c.get(1)); c.put(3, 3)
nhat_ky.append(c.get(2)); c.put(4, 4); nhat_ky += [c.get(1), c.get(3), c.get(4)]
print("get trả về:", nhat_ky, "| thứ tự hiện tại:", c.order)

get trả về: [1, -1, -1, 3, 4] | thứ tự hiện tại: [3, 4]


Muốn **mọi thao tác `O(1)`** ta cần hai thứ cùng lúc:

1. Tìm mục theo khóa trong `O(1)` → **hash map** `key → nút`.
2. Đưa một mục bất kỳ về "mới nhất" và bỏ mục "cũ nhất" trong `O(1)` → **danh sách liên kết đôi**.

### Danh sách liên kết đôi (doubly linked list)

Mỗi nút có **hai** con trỏ: `prev` (nút trước) và `next` (nút sau). Nhờ có `prev`, khi đang cầm một nút ta tháo nó ra khỏi danh sách trong `O(1)` mà không cần tìm nút đứng trước.

Dùng hai **nút canh** (sentinel) `head` và `tail` ở hai đầu để mọi nút thật luôn có hàng xóm — không cần xét trường hợp rỗng/đầu/cuối.

```text
head ⇄ [k=1] ⇄ [k=3] ⇄ tail
 ↑ cũ nhất ở ngay sau head        mới nhất ở ngay trước tail ↑

tháo nút x:     x.prev.next = x.next ;  x.next.prev = x.prev
chèn x trước tail:  x.prev = tail.prev ; x.next = tail ; tail.prev.next = x ; tail.prev = x
```

In [6]:
class NutDoi:
    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None

class DanhSachDoi:
    def __init__(self):
        self.head, self.tail = NutDoi(), NutDoi()      # nút canh
        self.head.next, self.tail.prev = self.tail, self.head

    def thao(self, x):                                 # O(1)
        x.prev.next = x.next
        x.next.prev = x.prev

    def them_cuoi(self, x):                            # O(1), chèn trước tail
        x.prev, x.next = self.tail.prev, self.tail
        self.tail.prev.next = x
        self.tail.prev = x

    def keys(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.key); cur = cur.next
        return out

d = DanhSachDoi()
nodes = {k: NutDoi(k, k * 10) for k in [1, 2, 3]}
for k in [1, 2, 3]:
    d.them_cuoi(nodes[k])
print("ban đầu:", d.keys())
d.thao(nodes[1]); d.them_cuoi(nodes[1])
print("dùng 1 → đưa về cuối:", d.keys())
cu_nhat = d.head.next; d.thao(cu_nhat)
print("đuổi nút cũ nhất", cu_nhat.key, "→", d.keys())

ban đầu: [1, 2, 3]
dùng 1 → đưa về cuối: [2, 3, 1]
đuổi nút cũ nhất 2 → [3, 1]


### Ghép lại: LRU `O(1)` = hash map + danh sách đôi

```text
get(key):  không có → -1 ; có → tháo nút, thêm cuối, trả val
put(key):  có → cập nhật val, tháo, thêm cuối
           chưa có → nếu đầy: tháo head.next, xóa khỏi map ; tạo nút, thêm cuối, ghi vào map
```

Nút phải lưu cả `key` để khi đuổi nút cũ nhất ta biết xóa khóa nào khỏi map.

In [7]:
class LRUMau:
    def __init__(self, capacity):
        self.cap, self.map, self.ds = capacity, {}, DanhSachDoi()

    def get(self, key):
        if key not in self.map:
            return -1
        x = self.map[key]
        self.ds.thao(x); self.ds.them_cuoi(x)
        return x.val

    def put(self, key, value):
        if key in self.map:
            x = self.map[key]; x.val = value
            self.ds.thao(x); self.ds.them_cuoi(x)
            return
        if len(self.map) == self.cap:
            cu = self.ds.head.next
            self.ds.thao(cu); del self.map[cu.key]
        x = NutDoi(key, value)
        self.ds.them_cuoi(x); self.map[key] = x

c = LRUMau(2)
kq = []
c.put(1, 1); c.put(2, 2); kq.append(c.get(1)); c.put(3, 3)
kq.append(c.get(2)); c.put(4, 4); kq += [c.get(1), c.get(3), c.get(4)]
print("get trả về:", kq, "| thứ tự:", c.ds.keys())

get trả về: [1, -1, -1, 3, 4] | thứ tự: [3, 4]


### Cách ngắn trong Python: `OrderedDict`

`collections.OrderedDict` vốn là dict nhớ thứ tự chèn, có `move_to_end(key)` và `popitem(last=False)` đều `O(1)` — bên trong chính là hash map + danh sách đôi.

Python còn có decorator `functools.lru_cache` để tự động cache kết quả hàm theo chính sách LRU.

In [8]:
from collections import OrderedDict
from functools import lru_cache

od = OrderedDict()
od[1] = "a"; od[2] = "b"; od[3] = "c"
od.move_to_end(1)                  # 1 thành mới nhất
print("thứ tự:", list(od), "| đuổi cũ nhất:", od.popitem(last=False))

@lru_cache(maxsize=2)
def binh_phuong(x):
    print(f"   (tính {x}²)")
    return x * x
for x in [2, 3, 2, 4, 3]:
    binh_phuong(x)
print(binh_phuong.cache_info())

thứ tự: [2, 3, 1] | đuổi cũ nhất: (2, 'b')
   (tính 2²)
   (tính 3²)
   (tính 4²)
   (tính 3²)
CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)


Ở dãy gọi `2, 3, 2, 4, 3` với `maxsize = 2`: lần gọi `2` thứ hai trúng cache; gọi `4` đuổi `3` (lâu nhất chưa dùng) nên gọi `3` lần cuối phải tính lại.

## 2.3 Áp dụng vào bài này

Đây chính là cấu trúc **LRU** ở trên: **hash map** `key → nút` để tìm `O(1)`, **danh sách liên kết đôi** với hai nút canh để đưa nút vừa dùng về cuối và đuổi nút cũ nhất ở đầu trong `O(1)`.

# 3. Baseline: list theo thứ tự dùng (`O(n)`)

Giữ một `dict` cho giá trị và một `list` các khóa theo thứ tự dùng. Mỗi lần dùng: `remove(key)` rồi `append(key)` — `remove` tốn `O(n)`.

In [9]:
from typing import Optional
from collections import OrderedDict
import random, time


class LRUList:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.data = {}
        self.order = []                     # cũ -> mới

    def get(self, key: int) -> int:
        if key not in self.data:
            return -1
        self.order.remove(key)              # O(n)
        self.order.append(key)
        return self.data[key]

    def put(self, key: int, value: int) -> None:
        if key in self.data:
            self.order.remove(key)
        elif len(self.data) == self.cap:
            old = self.order.pop(0)         # O(n)
            del self.data[old]
        self.data[key] = value
        self.order.append(key)


def replay(cls, ops, args):
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == "LRUCache":
            obj = cls(*a); out.append(None)
        elif op == "get":
            out.append(obj.get(*a))
        else:
            obj.put(*a); out.append(None)
    return out

OPS = ["LRUCache", "put", "put", "get", "put", "get", "put", "get", "get", "get"]
ARGS = [[2], [1, 1], [2, 2], [1], [3, 3], [2], [4, 4], [1], [3], [4]]
print(replay(LRUList, OPS, ARGS))

[None, None, None, 1, None, -1, None, -1, 3, 4]


# 4. Ý tưởng chính: hash map + danh sách liên kết đôi

| Thao tác cần | Cấu trúc làm được `O(1)` |
|---|---|
| Tìm nút theo khóa | hash map `key → node` |
| Xóa một nút **ở giữa** | danh sách **đôi** (biết `prev` và `next`) |
| Đưa nút lên "mới nhất" | thêm vào một đầu danh sách |
| Bỏ nút "cũ nhất" | xóa ở đầu kia |

Danh sách **đơn** không đủ: xóa một nút ở giữa cần biết nút **trước** nó.

**Nút canh:** hai nút giả `head` (phía cũ) và `tail` (phía mới), luôn tồn tại. Mọi nút thật nằm giữa chúng → thêm/xóa không bao giờ phải kiểm tra `None`.

```text
head <-> [cũ nhất] <-> ... <-> [mới nhất] <-> tail
```

In [10]:
class DNode:
    __slots__ = ("key", "val", "prev", "next")

    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None


class LRUManual:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.map = {}
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node: DNode) -> None:
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_tail(self, node: DNode) -> None:
        node.prev, node.next = self.tail.prev, self.tail
        self.tail.prev.next = node
        self.tail.prev = node

    def get(self, key: int) -> int:
        node = self.map.get(key)
        if not node:
            return -1
        self._remove(node)
        self._add_to_tail(node)
        return node.val

    def put(self, key: int, value: int) -> None:
        node = self.map.get(key)
        if node:
            node.val = value
            self._remove(node)
            self._add_to_tail(node)
            return
        if len(self.map) == self.cap:
            lru = self.head.next
            self._remove(lru)
            del self.map[lru.key]          # cần lưu key trong nút để xóa khỏi map
        node = DNode(key, value)
        self.map[key] = node
        self._add_to_tail(node)

    def keys_old_to_new(self):
        out, n = [], self.head.next
        while n is not self.tail:
            out.append(n.key); n = n.next
        return out

print(replay(LRUManual, OPS, ARGS))

[None, None, None, 1, None, -1, None, -1, 3, 4]


Nút lưu cả **`key`** (không chỉ `val`): khi đuổi nút cũ nhất, ta cần `key` để xóa nó khỏi hash map.

# 5. Mô phỏng

In [11]:
def trace(ops, args):
    obj = None
    for op, a in zip(ops, args):
        if op == "LRUCache":
            obj = LRUManual(*a)
            print(f"LRUCache({a[0]})")
            continue
        if op == "get":
            r = obj.get(*a)
            print(f"get({a[0]}) -> {r:>2}   thứ tự cũ→mới: {obj.keys_old_to_new()}")
        else:
            before = set(obj.map)
            obj.put(*a)
            evicted = before - set(obj.map)
            msg = f"  (đuổi {evicted.pop()})" if evicted else ""
            print(f"put({a[0]}, {a[1]})       thứ tự cũ→mới: {obj.keys_old_to_new()}{msg}")

trace(OPS, ARGS)

LRUCache(2)
put(1, 1)       thứ tự cũ→mới: [1]
put(2, 2)       thứ tự cũ→mới: [1, 2]
get(1) ->  1   thứ tự cũ→mới: [2, 1]
put(3, 3)       thứ tự cũ→mới: [1, 3]  (đuổi 2)
get(2) -> -1   thứ tự cũ→mới: [1, 3]
put(4, 4)       thứ tự cũ→mới: [3, 4]  (đuổi 1)
get(1) -> -1   thứ tự cũ→mới: [3, 4]
get(3) ->  3   thứ tự cũ→mới: [4, 3]
get(4) ->  4   thứ tự cũ→mới: [3, 4]


# 6. Cách gọn: `OrderedDict`

`OrderedDict` của Python **chính là** hash map + danh sách liên kết đôi bên trong:

- `move_to_end(key)` — đưa lên "mới nhất", `O(1)`.
- `popitem(last=False)` — bỏ phần tử "cũ nhất", `O(1)`.

In [12]:
class LRUOrdered:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.od = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.od:
            return -1
        self.od.move_to_end(key)
        return self.od[key]

    def put(self, key: int, value: int) -> None:
        if key in self.od:
            self.od.move_to_end(key)
        self.od[key] = value
        if len(self.od) > self.cap:
            self.od.popitem(last=False)

print(replay(LRUOrdered, OPS, ARGS))

[None, None, None, 1, None, -1, None, -1, 3, 4]


Trong phỏng vấn, nên biết **cả hai**: `OrderedDict` để viết nhanh, và bản tự cài để chứng minh hiểu cơ chế (nhiều người phỏng vấn sẽ yêu cầu không dùng `OrderedDict`).

# 7. Phiên bản đúng format LeetCode

In [13]:
class LRUCache:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.map = {}
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node: DNode) -> None:
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_tail(self, node: DNode) -> None:
        node.prev, node.next = self.tail.prev, self.tail
        self.tail.prev.next = node
        self.tail.prev = node

    def get(self, key: int) -> int:
        node = self.map.get(key)
        if not node:
            return -1
        self._remove(node)
        self._add_to_tail(node)
        return node.val

    def put(self, key: int, value: int) -> None:
        node = self.map.get(key)
        if node:
            node.val = value
            self._remove(node)
            self._add_to_tail(node)
            return
        if len(self.map) == self.cap:
            lru = self.head.next
            self._remove(lru)
            del self.map[lru.key]
        node = DNode(key, value)
        self.map[key] = node
        self._add_to_tail(node)

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual}")


check("Ví dụ", replay(LRUCache, OPS, ARGS), [None, None, None, 1, None, -1, None, -1, 3, 4])
check("Cập nhật", replay(LRUCache, ["LRUCache", "put", "put", "put", "get"], [[1], [1, 1], [1, 5], [2, 2], [1]]), [None, None, None, None, -1])
check("Cap 1", replay(LRUCache, ["LRUCache", "put", "get", "put", "get", "get"], [[1], [2, 1], [2], [3, 2], [2], [3]]), [None, None, 1, None, -1, 2])

random.seed(146)
for _ in range(1000):
    cap = random.randint(1, 5)
    ops, args = ["LRUCache"], [[cap]]
    for _ in range(60):
        if random.random() < 0.5:
            ops.append("get"); args.append([random.randint(0, 8)])
        else:
            ops.append("put"); args.append([random.randint(0, 8), random.randint(0, 100)])
    expected = replay(LRUList, ops, args)
    assert replay(LRUCache, ops, args) == expected == replay(LRUOrdered, ops, args)
print("1000 chuỗi thao tác ngẫu nhiên: 3 cài đặt khớp nhau")

ops, args = ["LRUCache"], [[3000]]
for _ in range(2 * 10**5):
    if random.random() < 0.5:
        ops.append("get"); args.append([random.randint(0, 10**4)])
    else:
        ops.append("put"); args.append([random.randint(0, 10**4), random.randint(0, 10**5)])
for cls in (LRUCache, LRUOrdered):
    t0 = time.time(); replay(cls, ops, args); t1 = time.time()
    print(f"{cls.__name__:<11} 2·10^5 thao tác: {t1 - t0:.2f}s")
t0 = time.time(); replay(LRUList, ops[:20001], args[:20001]); t1 = time.time()
print(f"{'LRUList':<11} chỉ 2·10^4 thao tác: {t1 - t0:.2f}s  (O(n) mỗi thao tác)")

Ví dụ          PASS  [None, None, None, 1, None, -1, None, -1, 3, 4]
Cập nhật       PASS  [None, None, None, None, -1]
Cap 1          PASS  [None, None, 1, None, -1, 2]


1000 chuỗi thao tác ngẫu nhiên: 3 cài đặt khớp nhau


LRUCache    2·10^5 thao tác: 0.49s


LRUOrdered  2·10^5 thao tác: 0.32s


LRUList     chỉ 2·10^4 thao tác: 0.26s  (O(n) mỗi thao tác)


# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:**

1. `map` chứa đúng các khóa đang trong cache, mỗi khóa trỏ tới nút của nó.
2. Danh sách `head → … → tail` chứa đúng các nút đó, theo thứ tự **lần dùng gần nhất tăng dần** (cũ ở gần `head`, mới ở gần `tail`).

`get`/`put` mỗi lần dùng một khóa đều chuyển nút của nó về sát `tail` → giữ bất biến 2. Khi đầy, nút cũ nhất là `head.next` — xóa khỏi cả danh sách và map → giữ bất biến 1.

```text
get, put: O(1) (trung bình, do hash map)
Space:    O(capacity)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: `get` không cập nhật thứ tự

In [15]:
class WrongGetNoTouch(LRUOrdered):
    def get(self, key):
        return self.od.get(key, -1)                 # SAI: không move_to_end

print("Sai :", replay(WrongGetNoTouch, OPS, ARGS))
print("Đúng:", replay(LRUCache, OPS, ARGS))

Sai : [None, None, None, 1, None, 2, None, -1, 3, 4]
Đúng: [None, None, None, 1, None, -1, None, -1, 3, 4]


Sau `get(1)`, khóa 1 phải thành "mới nhất"; bản sai vẫn coi 1 là cũ nhất và đuổi nó khi `put(3, 3)`.

## Lỗi 2: `put` khóa đã có mà vẫn đuổi một khóa khác

In [16]:
class WrongEvictOnUpdate(LRUOrdered):
    def put(self, key, value):
        if len(self.od) == self.cap:                # SAI: kiểm tra đầy trước khi xét khóa đã có
            self.od.popitem(last=False)
        self.od[key] = value
        self.od.move_to_end(key)

ops2 = ["LRUCache", "put", "put", "put", "get"]
args2 = [[2], [1, 1], [2, 2], [2, 20], [1]]
print("Sai :", replay(WrongEvictOnUpdate, ops2, args2), "| Đúng:", replay(LRUCache, ops2, args2))

Sai : [None, None, None, None, -1] | Đúng: [None, None, None, None, 1]


## Lỗi 3: Nút không lưu `key`

Khi đuổi `head.next`, không biết khóa nào để `del map[key]` → map giữ khóa "ma" trỏ tới nút đã bị gỡ.

## Lỗi 4: Dùng danh sách **đơn**

Không có `prev` → xóa một nút ở giữa phải duyệt từ đầu, `O(n)`.

# 10. Mẹo nhận diện pattern

**"Truy cập theo khóa + duy trì thứ tự / độ ưu tiên có thể thay đổi, mọi thao tác `O(1)`"** → hash map trỏ vào các nút của danh sách liên kết đôi.

Bài liên quan: **#460 LFU Cache** (Hard — thêm map tần suất → danh sách), **#432 All O`one Data Structure** (Hard), **#355 Design Twitter**, **#1472 Design Browser History**, **#2502 Design Memory Allocator**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`LRUCache(3)`; thao tác: `put(1,1), put(2,2), put(3,3), get(1), put(4,4), get(2), get(3), put(5,5), get(1), get(4)`. Kết quả của các `get`?

## Bài 2 — Điền chỗ trống

In [17]:
class exercise_2_LRU:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.od = OrderedDict()

    def get(self, key: int) -> int:
        # TODO
        pass

    def put(self, key: int, value: int) -> None:
        # TODO
        pass


# assert replay(exercise_2_LRU, OPS, ARGS) == [None, None, None, 1, None, -1, None, -1, 3, 4]

## Bài 3 — Biến thể: thêm `peek(key)` không làm thay đổi thứ tự, và `size()`

Mở rộng `LRUManual` thành `LRUWithPeek`.

In [18]:
class exercise_3_LRUWithPeek(LRUManual):
    def peek(self, key: int) -> int:
        # TODO
        pass

    def size(self) -> int:
        # TODO
        pass


# c = exercise_3_LRUWithPeek(2); c.put(1, 1); c.put(2, 2); assert c.peek(1) == 1; c.put(3, 3); assert c.get(1) == -1 and c.size() == 2

## Bài 4 — Biến thể: cache có thời hạn (TTL)

`put(key, value, now, ttl)`: khóa hết hạn ở thời điểm `now + ttl`. `get(key, now)` trả `-1` nếu khóa không có **hoặc đã hết hạn** (và xóa nó). Vẫn giữ chính sách LRU khi đầy.

In [19]:
class exercise_4_TTLCache:
    def __init__(self, capacity: int):
        # TODO
        pass

    def get(self, key: int, now: int) -> int:
        # TODO
        pass

    def put(self, key: int, value: int, now: int, ttl: int) -> None:
        # TODO
        pass


# c = exercise_4_TTLCache(2); c.put(1, 10, now=0, ttl=5); assert c.get(1, now=4) == 10 and c.get(1, now=5) == -1
# c = exercise_4_TTLCache(1); c.put(1, 1, 0, 100); c.put(2, 2, 1, 100); assert c.get(1, 2) == -1 and c.get(2, 2) == 2

## Bài 5 — Mini challenge: LFU Cache (LeetCode #460, Hard)

Khi đầy, đuổi khóa có **tần suất dùng nhỏ nhất**; hòa thì đuổi khóa **lâu nhất chưa dùng** trong số đó. `O(1)` mỗi thao tác.

Gợi ý nhanh: `key → (value, freq)`; `freq → OrderedDict các khóa`; biến `min_freq`.

In [20]:
class exercise_5_LFUCache:
    def __init__(self, capacity: int):
        # TODO
        pass

    def get(self, key: int) -> int:
        # TODO
        pass

    def put(self, key: int, value: int) -> None:
        # TODO
        pass


# assert replay(exercise_5_LFUCache, ["LRUCache", "put", "put", "get", "put", "get", "get", "put", "get", "get", "get"], [[2], [1, 1], [2, 2], [1], [3, 3], [2], [3], [4, 4], [1], [3], [4]]) == [None, None, None, 1, None, -1, 3, None, -1, 3, 4]

# 12. Gợi ý

**Bài 1**
- Mức 1: dùng `trace` với danh sách thao tác tương ứng.

**Bài 3**
- Mức 1: `peek`: `node = self.map.get(key)`; trả `node.val` hoặc `-1`, **không** gọi `_remove/_add_to_tail`.

**Bài 4**
- Mức 1: `OrderedDict key → (value, expire)`; `get` kiểm tra `now >= expire` thì xóa và trả `-1`.

**Bài 5**
- Mức 1: hàm `_touch(key)`: lấy `f`, xóa khỏi `buckets[f]`, nếu bucket rỗng và `f == min_freq` thì `min_freq += 1`; thêm vào `buckets[f + 1]`.
- Mức 2: `put` khóa mới: nếu đầy, `buckets[min_freq].popitem(last=False)`; rồi `min_freq = 1`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
ops1 = ["LRUCache", "put", "put", "put", "get", "put", "get", "get", "put", "get", "get"]
args1 = [[3], [1, 1], [2, 2], [3, 3], [1], [4, 4], [2], [3], [5, 5], [1], [4]]
trace(ops1, args1)

LRUCache(3)
put(1, 1)       thứ tự cũ→mới: [1]
put(2, 2)       thứ tự cũ→mới: [1, 2]
put(3, 3)       thứ tự cũ→mới: [1, 2, 3]
get(1) ->  1   thứ tự cũ→mới: [2, 3, 1]
put(4, 4)       thứ tự cũ→mới: [3, 1, 4]  (đuổi 2)
get(2) -> -1   thứ tự cũ→mới: [3, 1, 4]
get(3) ->  3   thứ tự cũ→mới: [1, 4, 3]
put(5, 5)       thứ tự cũ→mới: [4, 3, 5]  (đuổi 1)
get(1) -> -1   thứ tự cũ→mới: [4, 3, 5]
get(4) ->  4   thứ tự cũ→mới: [3, 5, 4]


## Đáp án Bài 2

In [22]:
class answer_2_LRU(LRUOrdered):
    pass

assert replay(answer_2_LRU, OPS, ARGS) == [None, None, None, 1, None, -1, None, -1, 3, 4]
print("Bài 2 PASS (giống LRUOrdered ở mục 6)")

Bài 2 PASS (giống LRUOrdered ở mục 6)


## Đáp án Bài 3

In [23]:
class answer_3_LRUWithPeek(LRUManual):
    def peek(self, key: int) -> int:
        node = self.map.get(key)
        return node.val if node else -1

    def size(self) -> int:
        return len(self.map)

c = answer_3_LRUWithPeek(2); c.put(1, 1); c.put(2, 2); assert c.peek(1) == 1; c.put(3, 3); assert c.get(1) == -1 and c.size() == 2
print("Bài 3 PASS (peek không làm 1 thành mới nhất nên 1 vẫn bị đuổi)")

Bài 3 PASS (peek không làm 1 thành mới nhất nên 1 vẫn bị đuổi)


## Đáp án Bài 4

In [24]:
class answer_4_TTLCache:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.od = OrderedDict()

    def get(self, key: int, now: int) -> int:
        if key not in self.od:
            return -1
        value, expire = self.od[key]
        if now >= expire:
            del self.od[key]
            return -1
        self.od.move_to_end(key)
        return value

    def put(self, key: int, value: int, now: int, ttl: int) -> None:
        if key in self.od:
            self.od.move_to_end(key)
        self.od[key] = (value, now + ttl)
        if len(self.od) > self.cap:
            self.od.popitem(last=False)

c = answer_4_TTLCache(2); c.put(1, 10, now=0, ttl=5); assert c.get(1, now=4) == 10 and c.get(1, now=5) == -1
c = answer_4_TTLCache(1); c.put(1, 1, 0, 100); c.put(2, 2, 1, 100); assert c.get(1, 2) == -1 and c.get(2, 2) == 2
print("Bài 4 PASS")

Bài 4 PASS

(Bản này chỉ dọn khóa hết hạn khi bị `get` tới — "lazy expiration". Một khóa hết hạn vẫn chiếm chỗ cho tới khi bị đuổi theo LRU. Hệ thống thật thường kết hợp thêm dọn định kỳ.)

## Đáp án Bài 5

In [25]:
from collections import defaultdict

class answer_5_LFUCache:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.kv = {}                                   # key -> [value, freq]
        self.buckets = defaultdict(OrderedDict)        # freq -> keys (cũ -> mới)
        self.min_freq = 0

    def _touch(self, key):
        value, f = self.kv[key]
        del self.buckets[f][key]
        if not self.buckets[f]:
            del self.buckets[f]
            if self.min_freq == f:
                self.min_freq += 1
        self.kv[key][1] = f + 1
        self.buckets[f + 1][key] = None

    def get(self, key: int) -> int:
        if key not in self.kv:
            return -1
        self._touch(key)
        return self.kv[key][0]

    def put(self, key: int, value: int) -> None:
        if self.cap == 0:
            return
        if key in self.kv:
            self.kv[key][0] = value
            self._touch(key)
            return
        if len(self.kv) == self.cap:
            old, _ = self.buckets[self.min_freq].popitem(last=False)
            if not self.buckets[self.min_freq]:
                del self.buckets[self.min_freq]
            del self.kv[old]
        self.kv[key] = [value, 1]
        self.buckets[1][key] = None
        self.min_freq = 1

assert replay(answer_5_LFUCache, ["LRUCache", "put", "put", "get", "put", "get", "get", "put", "get", "get", "get"], [[2], [1, 1], [2, 2], [1], [3, 3], [2], [3], [4, 4], [1], [3], [4]]) == [None, None, None, 1, None, -1, 3, None, -1, 3, 4]

class LFUBrute:
    def __init__(self, cap):
        self.cap, self.kv, self.freq, self.last, self.t = cap, {}, {}, {}, 0
    def _use(self, k):
        self.t += 1; self.freq[k] = self.freq.get(k, 0) + 1; self.last[k] = self.t
    def get(self, k):
        if k not in self.kv: return -1
        self._use(k); return self.kv[k]
    def put(self, k, v):
        if k not in self.kv and len(self.kv) == self.cap:
            victim = min(self.kv, key=lambda x: (self.freq[x], self.last[x]))
            del self.kv[victim], self.freq[victim], self.last[victim]
        self.kv[k] = v; self._use(k)

for _ in range(1000):
    cap = random.randint(1, 4)
    ops, args = ["LRUCache"], [[cap]]
    for _ in range(50):
        if random.random() < 0.5:
            ops.append("get"); args.append([random.randint(0, 6)])
        else:
            ops.append("put"); args.append([random.randint(0, 6), random.randint(0, 50)])
    assert replay(answer_5_LFUCache, ops, args) == replay(LFUBrute, ops, args)
print("Bài 5 PASS (khớp bản brute force trên 1000 chuỗi thao tác)")

Bài 5 PASS (khớp bản brute force trên 1000 chuỗi thao tác)


# 14. Tổng kết

> Hash map `key → nút`, danh sách liên kết **đôi** với hai nút canh giữ thứ tự dùng (cũ ở `head`, mới ở `tail`). Mỗi lần dùng: gỡ nút, gắn về `tail`. Đầy thì đuổi `head.next` và xóa khỏi map (nút phải lưu `key`). Mọi thao tác `O(1)`.

---

## Checklist trước khi nộp

- [ ] `get` cũng đưa khóa lên "mới nhất"?
- [ ] `put` khóa đã có: cập nhật + đưa lên, **không** đuổi?
- [ ] Nút lưu `key` để xóa khỏi map khi đuổi?
- [ ] Danh sách **đôi** + nút canh?